# Slide-based cryptanalysis of 4-round DES (Solution 2)

This notebook implements the **two-characteristic slide strategy** from *DES Cryptanalysis* slides (Figure 3, slides 13–15):

- **Ω₁** (`P_diff = 0x2000000000000000`): round-1 probability-1 structure with `R0'=0`; only **S1** is active in round 2. The round-2 Feistel output difference `B'` has only the **S1 output nibble** non-zero → **16** candidates `B' = P(s1_dout ≪ 28)`.
- **Ω₂** (`P_diff = 0x0222222200000000`): symmetric layout; **S2–S8** active in round 2, used to recover the **S1** key bits of `K4`.

### Slide notation vs ciphertext halves

The slides write `C = (l || r)` at the 4-round output with **`l = R4`** (left) and **`r = L4 = R3`** (right). With chosen pairs differing only in the left half (`R0'=0`):

| Symbol | Meaning | In code |
|--------|---------|---------|
| `l'` | Left half of ciphertext XOR | `slide_l = R4'` |
| `r'` | Right half / round-4 F input diff `d'` | `slide_r = L4' = R3'` |
| (8) | `D' = B' ⊕ l'` | `Dprime = Bprime ^ slide_l` |

**Common pitfall:** treating slide `l'` as **`L4'`** (the Feistel `L4` register) breaks `D'` — you must XOR with **`R4'`** (left ciphertext half).

Ω₁ leaves **S4** with zero input difference in round 4 (`E(L4')` inactive for S4); we **brute-force the 6-bit S4 chunk** of `K4` (64 trials) after Ω₁+Ω₂, then invert the key schedule like Solution 1.


In [5]:
## Setup: helpers and characteristic expansion check

In [6]:
import collections
import os
import random
import sys

for _cand in ('.', '..', os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(_cand, 'oracle_app')):
        sys.path.insert(0, os.path.abspath(_cand))
        break

from oracle_app.custom_DES import (
    S_BOXES, E, P, PC1, PC2, SHIFT_SCHEDULE,
    des_encrypt_hex, permute, left_shift, to_bin,
)

OMEGA1 = int('2000000000000000', 16)
OMEGA2 = int('0222222200000000', 16)
NUM_PAIRS = int(os.environ.get('SLIDE_NUM_PAIRS', '40'))
random.seed(42)

def bits_of(val, n):
    return [(val >> (n - 1 - i)) & 1 for i in range(n)]

def from_bits(bits):
    v = 0
    for b in bits:
        v = (v << 1) | b
    return v

def expand(R):
    rb = bits_of(R, 32)
    return from_bits([rb[E[k] - 1] for k in range(48)])

def permP(x):
    xb = bits_of(x, 32)
    return from_bits([xb[P[k] - 1] for k in range(32)])

Pinv = [0] * 32
for _i in range(32):
    Pinv[P[_i] - 1] = _i + 1

def permPinv(y):
    yb = bits_of(y, 32)
    return from_bits([yb[Pinv[k] - 1] for k in range(32)])

def sbox(idx, six):
    b = bits_of(six, 6)
    row = (b[0] << 1) | b[5]
    col = (b[1] << 3) | (b[2] << 2) | (b[3] << 1) | b[4]
    return S_BOXES[idx][row][col]

BPRIME_CANDS = [permP(s1_out << 28) for s1_out in range(16)]

def check_expansion(label, L0p):
    ed = bits_of(expand(L0p), 48)
    active = []
    for m in range(8):
        chunk = from_bits(ed[6 * m:6 * m + 6])
        if chunk:
            active.append(f'S{m+1}(0x{chunk:02x})')
    print(f'{label}: L0\'=0x{L0p:08X}  -> round-2 active: {", ".join(active) or "none"}')

check_expansion('Omega1 (Figure 3, left)', OMEGA1 >> 32)
check_expansion('Omega2 (Figure 3, right)', OMEGA2 >> 32)
print('helpers ready')


Omega1 (Figure 3, left): L0'=0x20000000  -> round-2 active: S1(0x04)
Omega2 (Figure 3, right): L0'=0x02222222  -> round-2 active: S2(0x04), S3(0x04), S4(0x04), S5(0x04), S6(0x04), S7(0x04), S8(0x04)
helpers ready


In [7]:
import requests

BASE_URL = os.environ.get('ORACLE_URL', 'http://localhost').rstrip('/')
_query_count = 0
_session = requests.Session()

def oracle(pt64):
    global _query_count
    _query_count += 1
    resp = _session.post(
        f'{BASE_URL}/api/encrypt',
        json={'plaintext': f'{pt64:016X}'},
        timeout=60,
    )
    if resp.status_code == 429:
        raise RuntimeError(
            'Oracle rate limit (1000/day per IP). Wait until the quota resets or use another network.'
        )
    resp.raise_for_status()
    return int(resp.json()['ciphertext'], 16)

if os.environ.get('SKIP_ORACLE_PROBE', '').lower() not in ('1', 'true', 'yes'):
    _probe = oracle(0x0123456789ABCDEF)
    print(f'Oracle {BASE_URL} OK; sample CT = {_probe:016X}')
else:
    print(f'Oracle URL: {BASE_URL} (probe skipped; set SKIP_ORACLE_PROBE=1 to save 1 query)')


Oracle http://localhost OK; sample CT = F50804444FF5DE47


## Ω₁ — recover `K4` bits for S2–S8 (slide 13–14)

For each pair `(P, P⊕Ω₁)` and each `B'` candidate, compute `D' = B'⊕R4'`, derive S-box output nibble differences, and vote on 6-bit `K4` chunks for **S2…S8** (indices 1–7). Pick the `B'` with largest vote margin on those boxes.


In [8]:
def ct_halves(C, Cs):
    R4, L4 = C >> 32, C & 0xFFFFFFFF
    R4s, L4s = Cs >> 32, Cs & 0xFFFFFFFF
    return R4 ^ R4s, L4 ^ L4s, L4, L4s

def add_votes(C, Cs, Bprime, boxes, counts):
    slide_l, slide_r, L4, L4s = ct_halves(C, Cs)
    Dprime = Bprime ^ slide_l
    sob = bits_of(permPinv(Dprime), 32)
    idb = bits_of(expand(slide_r), 48)
    in_a = bits_of(expand(L4), 48)
    in_b = bits_of(expand(L4s), 48)
    for m in boxes:
        din = from_bits(idb[6 * m: 6 * m + 6])
        if din == 0:
            continue
        dout = from_bits(sob[4 * m: 4 * m + 4])
        a = from_bits(in_a[6 * m: 6 * m + 6])
        b = from_bits(in_b[6 * m: 6 * m + 6])
        for k in range(64):
            if sbox(m, a ^ k) ^ sbox(m, b ^ k) == dout:
                counts[m][k] += 1

def vote_margin(counts, boxes):
    margin = 0
    for m in boxes:
        top = counts[m].most_common(2)
        if not top:
            continue
        margin += top[0][1] - (top[1][1] if len(top) > 1 else 0)
    return margin

o1 = {B: [collections.Counter() for _ in range(8)] for B in BPRIME_CANDS}
for _ in range(NUM_PAIRS):
    pt = random.getrandbits(64)
    C, Cs = oracle(pt), oracle(pt ^ OMEGA1)
    for B in BPRIME_CANDS:
        add_votes(C, Cs, B, list(range(1, 8)), o1[B])

B1 = max(BPRIME_CANDS, key=lambda B: vote_margin(o1[B], list(range(1, 8))))
print(f"Ω₁ best B' = 0x{B1:08X}")
for m in range(1, 8):
    top = o1[B1][m].most_common(2)
    if not top:
        print(f'  S{m+1}: inactive (din=0)')
        continue
    print(f'  S{m+1}: K4 chunk 0x{top[0][0]:02x} votes={top[0][1]}')


Ω₁ best B' = 0x00000000
  S2: K4 chunk 0x16 votes=38
  S3: K4 chunk 0x39 votes=36
  S4: K4 chunk 0x21 votes=32
  S5: K4 chunk 0x26 votes=35
  S6: K4 chunk 0x3a votes=36
  S7: K4 chunk 0x1f votes=39
  S8: K4 chunk 0x23 votes=31


## Ω₂ — recover `K4` bits for S1 (slide 15)

Same slide equation, but Ω₂ makes **S1** the informative S-box in round 4; we vote only on **S1** (index 0).


In [9]:
o2 = {B: [collections.Counter() for _ in range(8)] for B in BPRIME_CANDS}
for _ in range(NUM_PAIRS):
    pt = random.getrandbits(64)
    C, Cs = oracle(pt), oracle(pt ^ OMEGA2)
    for B in BPRIME_CANDS:
        add_votes(C, Cs, B, [0], o2[B])

B2 = max(BPRIME_CANDS, key=lambda B: vote_margin(o2[B], [0]))
s1_chunk = o2[B2][0].most_common(1)[0][0]
print(f"Ω₂ best B' = 0x{B2:08X}; S1 K4 chunk = 0x{s1_chunk:02x}")


Ω₂ best B' = 0x00000000; S1 K4 chunk = 0x22


## Full key recovery

Assemble `K4`, brute the missing **S4** chunk (Ω₁ gives `din=0` for S4), invert the 4-round key schedule + **256**-way search on the eight unknown key-schedule bits, then verify against the oracle.


In [10]:
def rotr_list(bits, n):
    return bits[-n:] + bits[:-n]

TOTAL_SHIFT = sum(SHIFT_SCHEDULE)

def subkeys_from_combined0(c0):
    left, right = c0[:28], c0[28:]
    subs = []
    for shift in SHIFT_SCHEDULE:
        left = left_shift(left, shift)
        right = left_shift(right, shift)
        subs.append(permute(left + right, PC2))
    return subs

def encrypt_with_combined0(pt64, c0):
    subs = subkeys_from_combined0(c0)
    block = [str(b) for b in bits_of(pt64, 64)]
    left, right = block[:32], block[32:]
    for sk in subs:
        exp = permute(right, E)
        xr = [str(int(exp[i]) ^ int(sk[i])) for i in range(48)]
        sub = []
        for i in range(8):
            ch = xr[i * 6:(i + 1) * 6]
            r = int(ch[0] + ch[5], 2)
            c = int(''.join(ch[1:5]), 2)
            sub.extend(list(to_bin(S_BOXES[i][r][c], 4)))
        f_out = permute(sub, P)
        right, left = [str(int(left[i]) ^ int(f_out[i])) for i in range(32)], right
    return int(''.join(right + left), 2)

chunks = [0] * 8
chunks[0] = s1_chunk
for m in range(1, 8):
    if m == 3:
        continue
    chunks[m] = o1[B1][m].most_common(1)[0][0]

kp_pt = random.getrandbits(64)
kp_ct = oracle(kp_pt)
recovered_key_hex = None
K4_rec = None

for s4 in range(64):
    chunks[3] = s4
    k4 = 0
    for m in range(8):
        k4 = (k4 << 6) | chunks[m]
    k4_bits = bits_of(k4, 48)
    combined4 = [None] * 56
    for k in range(48):
        combined4[PC2[k] - 1] = k4_bits[k]
    unknown_pos = [i for i in range(56) if combined4[i] is None]
    for guess in range(1 << len(unknown_pos)):
        c4 = list(combined4)
        for idx, pos in enumerate(unknown_pos):
            c4[pos] = (guess >> (len(unknown_pos) - 1 - idx)) & 1
        c0 = rotr_list(c4[:28], TOTAL_SHIFT) + rotr_list(c4[28:], TOTAL_SHIFT)
        c0 = [str(b) for b in c0]
        if encrypt_with_combined0(kp_pt, c0) == kp_ct:
            key64 = [0] * 64
            for k in range(56):
                key64[PC1[k] - 1] = int(c0[k])
            key56 = [str(key64[i]) for i in range(64) if (i + 1) % 8 != 0]
            recovered_key_hex = f"{int(''.join(key56), 2):014X}"
            K4_rec = k4
            break
    if recovered_key_hex:
        break

assert recovered_key_hex, 'schedule inversion failed'

ok = all(
    des_encrypt_hex(f'{(pt := random.getrandbits(64)):016X}', recovered_key_hex)
    == f'{oracle(pt):016X}'
    for _ in range(15)
)
print(f'Recovered K4 = {K4_rec:012X}')
print(f'Recovered oracle key = {recovered_key_hex}')
print(f'Verification: {"PASS" if ok else "FAIL"}')
print(f'Total oracle queries = {_query_count}')


Recovered K4 = 896E619BA7E3
Recovered oracle key = 8B2F88A663CB4D
Verification: PASS
Total oracle queries = 177
